# 📊 Importar KM Dinâmico - Multi Anos (2023-2026)

Script que:
- ✅ Processa anos 2023, 2024, 2025, 2026
- ✅ Cria tabela separada por ano
- ✅ Remove duplicados do dataframe
- ✅ Verifica registos já existentes
- ✅ Insere APENAS registos novos
- ✅ Usa UNIQUE constraint para evitar duplicados

In [ ]:
# ==========================
# 1. Imports
# ==========================

import os
import platform
import re
from datetime import datetime
from pathlib import Path

import duckdb
import openpyxl
import pandas as pd

print("✓ Imports carregados")


In [ ]:
# ==========================
# 2. Ano a importar
# ==========================

ANO_IMPORTACAO = 2026

print(f"Ano a importar: {ANO_IMPORTACAO}")


In [ ]:
# ==========================
# 3. Configuração
# ==========================

ANOS = [ANO_IMPORTACAO]

BASE_PATH_ROOT = Path(
    r"T:\Portugal\D-Trafico\KM BASE"
)

MONTH_FILES = {
    1: "01 JANEIRO.xlsx",
    2: "02 FEVEREIRO.xlsx",
    3: "03 MARÇO.xlsx",
    4: "04 ABRIL.xlsx",
    5: "05 MAIO.xlsx",
    6: "06 JUNHO.xlsx",
    7: "07 JULHO.xlsx",
    8: "08 AGOSTO.xlsx",
    9: "09 Setembro.xlsx",
    10: "10 Outubro.xlsx",
    11: "11 Novembro.xlsx",
    12: "12 Dezembro.xlsx",
}

if platform.system() == "Windows":
    DB_PATH = Path(
        r"C:\Users\LISARR\OneDrive - Salvesen Logística S.A\00.DB\2026.duckdb"
    )

elif platform.system() == "Darwin":
    DB_PATH = (
        Path.home()
        / "Library/Mobile Documents/com~apple~CloudDocs/05.Salvesen/00_DB/2026.duckdb"
    )

else:
    raise OSError(
        f"Sistema operativo não suportado: {platform.system()}"
    )

DB_PATH.parent.mkdir(
    parents=True,
    exist_ok=True,
)

with duckdb.connect(str(DB_PATH)) as con:
    con.execute("SELECT 1")

print(f"Base path: {BASE_PATH_ROOT}")
print(f"Anos a processar: {ANOS}")
print(f"Meses por ano: {len(MONTH_FILES)}")
print(f"BD: {DB_PATH}")


In [ ]:
# ── FUNÇÕES DE PARSING ────────────────────────────────────────────────────
def clean_value(val):
    """Limpar e converter valores."""
    if val is None or (isinstance(val, float) and pd.isna(val)):
        return 0
    if isinstance(val, (int, float)):
        return float(val)
    return 0

def parse_formula(formula_str):
    """Extrair tipo_contrato, km_gratis e rate_km da fórmula."""
    if not formula_str:
        return None, 0, 0
    
    tipo_formula = "desconhecido"
    km_gratis = 0
    rate = 0
    
    if "fixo" in str(formula_str).lower():
        tipo_formula = "fixo"
    elif "km" in str(formula_str).lower():
        tipo_formula = "km"
    
    if "-" in str(formula_str):
        try:
            parts = str(formula_str).split("-")
            if len(parts) >= 2:
                km_gratis = float(parts[1].replace(")", "").strip())
        except:
            pass
    
    return tipo_formula, km_gratis, rate

def parse_cell_a(cell_value, sheet_name, row_idx, anon_map):
    """Trator = XX-YY-ZZ (2-2-2 exatamente) ou AABBBB
    Reboque = tudo o resto
    Descrição = tipo, nomes"""
    if not cell_value:
        return sheet_name, None, None, None
    
    cell_str = str(cell_value).strip()
    transportador = sheet_name.strip()
    
    trator = None
    reboque = None
    
    matricula_std = r'([A-Z0-9]{2}-[A-Z0-9]{2}-[A-Z0-9]{2})'
    codigo_trator = r'(?<![A-Z0-9-])([A-Z]{2}\d{4,5})(?![A-Z0-9-])'
    
    tratores_candidatos = []
    
    for match in re.finditer(matricula_std, cell_str):
        m_str = match.group(1)
        partes = m_str.split('-')
        if len(partes) == 3 and len(partes[0]) == 2 and len(partes[1]) == 2 and len(partes[2]) == 2:
            tratores_candidatos.append((m_str, match.start(), 'std'))
    
    for match in re.finditer(codigo_trator, cell_str):
        tratores_candidatos.append((match.group(1), match.start(), 'cod'))
    
    tratores_candidatos.sort(key=lambda x: x[1])
    
    if tratores_candidatos:
        trator = tratores_candidatos[0][0]
    
    if len(tratores_candidatos) >= 2:
        reboque = tratores_candidatos[1][0]
    else:
        temp_str = cell_str
        if trator:
            temp_str = temp_str.replace(trator, '', 1)
        
        match_l = re.search(r'(L-?\d+)', temp_str)
        if match_l:
            reboque = match_l.group(1)
        else:
            match_cod = re.search(r'([A-Z]{1,2}-\d{4,5})', temp_str)
            if match_cod:
                reboque = match_cod.group(1)
    
    descricao = cell_str
    if trator:
        descricao = descricao.replace(trator, '', 1)
    if reboque:
        descricao = descricao.replace(reboque, '', 1)
    
    parts = descricao.split()
    if parts and len(parts[0]) < 10 and parts[0].isalpha():
        if len(parts[0]) < 6:
            descricao = ' '.join(parts[1:])
    
    descricao = re.sub(r'\s*\/\s*', ' ', descricao)
    descricao = re.sub(r'\(\s*\)', '', descricao)
    descricao = re.sub(r'\s+', ' ', descricao)
    descricao = descricao.strip()
    
    return transportador, descricao if descricao else None, trator, reboque


def extract_sheet_data(ws_values, ws_formulas, sheet_name, month):
    """Extrair dados da folha de cálculo."""
    sheet_data = []
    
    first_row = list(ws_values.iter_rows(min_row=1, max_row=1, values_only=True))[0]
    date_columns = {}
    
    for col_idx in range(2, len(first_row)):
        val = first_row[col_idx]
        if isinstance(val, datetime):
            date_columns[col_idx] = val
    
    if not date_columns:
        return sheet_data
    
    anon_map = {}
    row_idx = 2
    
    while row_idx <= ws_values.max_row:
        current_row = list(ws_values.iter_rows(
            min_row=row_idx, max_row=row_idx, values_only=True))[0]
        
        if not current_row or not current_row[0]:
            row_idx += 1
            continue
        
        col_b = str(current_row[1]).strip().upper() if current_row[1] else ""
        if col_b != "KM":
            row_idx += 1
            continue
        
        transportador, descricao, trator, reboque = parse_cell_a(
            current_row[0], sheet_name, row_idx, anon_map)
        
        # Usar trator ou reboque para determinar se é válido
        if not (trator or reboque):
            row_idx += 1
            continue
        
        km_row = current_row
        p_row = list(ws_values.iter_rows(min_row=row_idx+1, max_row=row_idx+1, values_only=True))[0]
        vt_row = list(ws_values.iter_rows(min_row=row_idx+2, max_row=row_idx+2, values_only=True))[0]
        sd_row = list(ws_values.iter_rows(min_row=row_idx+3, max_row=row_idx+3, values_only=True))[0]
        tot_row = list(ws_values.iter_rows(min_row=row_idx+4, max_row=row_idx+4, values_only=True))[0]
        
        formula_cell = ws_formulas.cell(row=row_idx + 3, column=3).value
        tipo_formula, km_gratis, rate = parse_formula(formula_cell)
        
        for col_idx, date_obj in date_columns.items():
            if col_idx >= len(km_row):
                continue
            
            km_value = clean_value(km_row[col_idx])
            if not km_value or km_value == 0:
                continue
            
            portagens = clean_value(p_row[col_idx] if col_idx < len(p_row) else None)
            valor_total = clean_value(vt_row[col_idx] if col_idx < len(vt_row) else None)
            sub_divisao = clean_value(sd_row[col_idx] if col_idx < len(sd_row) else None)
            preco_base = valor_total - sub_divisao if (valor_total and sub_divisao) else valor_total
            
            sheet_data.append({
                'transportador': transportador,
                'descricao': descricao,
                'trator': trator,
                'reboque': reboque,
                'data': date_obj,
                'dia': date_obj.day,
                'mes': month,
                'km': int(km_value) if km_value else 0,
                'portagens': portagens,
                'preco_base': preco_base,
                'sub_divisao': sub_divisao,
                'tipo_contrato': tipo_formula,
                'km_gratis': km_gratis,
                'rate_km': rate,
                'total': valor_total,
            })
        
        row_idx += 5
    
    return sheet_data

print("✅ Funções carregadas")

In [ ]:
# ==========================
# 5. Processar anos
# ==========================

with duckdb.connect(str(DB_PATH)) as con:

    for ano in ANOS:

        print(
            f"\n{'=' * 70}\n"
            f"PROCESSANDO ANO {ano}\n"
            f"{'=' * 70}\n"
        )

        BASE_PATH = (
            BASE_PATH_ROOT
            / f"Ano {ano}"
        )

        table_name = f"km_diario_{ano}"

        if not BASE_PATH.exists():
            print(
                f"Pasta não encontrada: "
                f"{BASE_PATH}\n"
            )
            continue

        # ==========================
        # 5.1. Garantir tabela
        # ==========================

        con.execute(f"""
            CREATE TABLE IF NOT EXISTS "{table_name}" (
                id BIGINT,
                transportador VARCHAR NOT NULL,
                descricao VARCHAR,
                tipo_veiculo VARCHAR,
                viatura VARCHAR,
                data DATE NOT NULL,
                dia INTEGER,
                mes INTEGER,
                km INTEGER,
                portagens DOUBLE,
                preco_base DOUBLE,
                sub_divisao DOUBLE,
                tipo_contrato VARCHAR,
                km_gratis DOUBLE,
                rate_km DOUBLE,
                total DOUBLE,
                criado_em TIMESTAMP DEFAULT CURRENT_TIMESTAMP
            )
        """)

        con.execute(
            f'CREATE UNIQUE INDEX IF NOT EXISTS '
            f'"idx_{table_name}_chave" '
            f'ON "{table_name}" '
            f'(transportador, tipo_veiculo, viatura, data)'
        )

        # ==========================
        # 5.2. Ler Excel
        # ==========================

        all_data = []

        for month, filename in MONTH_FILES.items():

            filepath = (
                BASE_PATH
                / filename
            )

            if not filepath.exists():
                continue

            try:
                wb_val = openpyxl.load_workbook(
                    filepath,
                    data_only=True,
                    read_only=False,
                )

                wb_form = openpyxl.load_workbook(
                    filepath,
                    data_only=False,
                    read_only=False,
                )

                data = []

                for sheet_name in wb_val.sheetnames:

                    ws_v = wb_val[sheet_name]
                    ws_f = wb_form[sheet_name]

                    sheet_data = extract_sheet_data(
                        ws_v,
                        ws_f,
                        sheet_name.strip(),
                        month,
                    )

                    data.extend(
                        sheet_data
                    )

                all_data.extend(
                    data
                )

                wb_val.close()
                wb_form.close()

                print(
                    f"{filename}: "
                    f"{len(data):,} registos"
                )

            except Exception as erro:
                print(
                    f"{filename}: "
                    f"ERRO — {erro}"
                )

        if not all_data:
            print(
                f"Nenhum dado encontrado "
                f"para {ano}\n"
            )
            continue

        # ==========================
        # 5.3. Preparar DataFrame
        # ==========================

        df = pd.DataFrame(
            all_data
        )

        df = df.rename(
            columns={
                "trator": "tipo_veiculo",
                "reboque": "viatura",
            }
        )

        df["data"] = pd.to_datetime(
            df["data"],
            errors="coerce",
        ).dt.date

        df_before = len(df)

        df = df.drop_duplicates(
            subset=[
                "transportador",
                "tipo_veiculo",
                "viatura",
                "data",
            ],
            keep="first",
        )

        duplicados_dataframe = (
            df_before - len(df)
        )

        print(
            f"Duplicados removidos do DataFrame: "
            f"{duplicados_dataframe:,}"
        )

        # ==========================
        # 5.4. Inserir apenas novos
        # ==========================

        antes = con.execute(
            f'SELECT COUNT(*) '
            f'FROM "{table_name}"'
        ).fetchone()[0]

        max_id = con.execute(f"""
            SELECT COALESCE(
                MAX(TRY_CAST(id AS BIGINT)),
                0
            )
            FROM "{table_name}"
        """).fetchone()[0]

        df_insert = df.copy()

        df_insert.insert(
            0,
            "id",
            range(
                int(max_id) + 1,
                int(max_id) + 1 + len(df_insert),
            ),
        )

        df_insert["criado_em"] = (
            pd.Timestamp.now()
        )

        colunas_bd = [
            "id",
            "transportador",
            "descricao",
            "tipo_veiculo",
            "viatura",
            "data",
            "dia",
            "mes",
            "km",
            "portagens",
            "preco_base",
            "sub_divisao",
            "tipo_contrato",
            "km_gratis",
            "rate_km",
            "total",
            "criado_em",
        ]

        df_insert = df_insert[
            colunas_bd
        ]

        con.register(
            "batch_km",
            df_insert,
        )

        try:
            con.execute(f"""
                INSERT OR IGNORE INTO "{table_name}"
                SELECT *
                FROM batch_km
            """)

        finally:
            con.unregister(
                "batch_km"
            )

        depois = con.execute(
            f'SELECT COUNT(*) '
            f'FROM "{table_name}"'
        ).fetchone()[0]

        inseridos = depois - antes
        ja_existentes = len(df) - inseridos

        print(
            f"Novos registos: {inseridos:,}"
        )

        print(
            f"Já existentes: {ja_existentes:,}"
        )

        print(
            f"Total em '{table_name}': "
            f"{depois:,}\n"
        )

    con.checkpoint()

print(
    f"{'=' * 70}\n"
    f"Processo concluído com sucesso.\n"
    f"{'=' * 70}"
)


In [ ]:
# ==========================
# 6. Validação final
# ==========================

with duckdb.connect(
    str(DB_PATH),
    read_only=True,
) as con:

    validacao = []
    duplicados = []

    for ano in ANOS:

        tabela = f"km_diario_{ano}"

        existe = con.execute("""
            SELECT COUNT(*)
            FROM information_schema.tables
            WHERE table_schema = 'main'
              AND table_name = ?
        """, [tabela]).fetchone()[0]

        if not existe:
            continue

        total, ids_unicos = con.execute(f"""
            SELECT
                COUNT(*),
                COUNT(DISTINCT id)
            FROM "{tabela}"
        """).fetchone()

        grupos_duplicados, linhas_duplicadas = con.execute(f"""
            SELECT
                COUNT(*),
                COALESCE(SUM(n), 0)
            FROM (
                SELECT
                    transportador,
                    tipo_veiculo,
                    viatura,
                    data,
                    COUNT(*) AS n
                FROM "{tabela}"
                GROUP BY
                    transportador,
                    tipo_veiculo,
                    viatura,
                    data
                HAVING COUNT(*) > 1
            )
        """).fetchone()

        validacao.append({
            "Tabela": tabela,
            "Linhas": total,
            "IDs_Unicos": ids_unicos,
            "Grupos_Duplicados": grupos_duplicados,
            "Linhas_Duplicadas": linhas_duplicadas,
            "OK": (
                total == ids_unicos
                and grupos_duplicados == 0
            ),
        })

        if grupos_duplicados > 0:

            df_dup = con.execute(f"""
                WITH chaves_duplicadas AS (
                    SELECT
                        transportador,
                        tipo_veiculo,
                        viatura,
                        data,
                        COUNT(*) AS qtd_duplicados
                    FROM "{tabela}"
                    GROUP BY
                        transportador,
                        tipo_veiculo,
                        viatura,
                        data
                    HAVING COUNT(*) > 1
                )

                SELECT
                    d.qtd_duplicados,
                    t.*
                FROM "{tabela}" t
                INNER JOIN chaves_duplicadas d
                    ON t.transportador
                        IS NOT DISTINCT FROM d.transportador
                   AND t.tipo_veiculo
                        IS NOT DISTINCT FROM d.tipo_veiculo
                   AND t.viatura
                        IS NOT DISTINCT FROM d.viatura
                   AND t.data
                        IS NOT DISTINCT FROM d.data
                ORDER BY
                    t.transportador,
                    t.tipo_veiculo,
                    t.viatura,
                    t.data,
                    TRY_CAST(t.id AS BIGINT)
            """).df()

            df_dup.insert(
                0,
                "Tabela",
                tabela,
            )

            duplicados.append(
                df_dup
            )

df_validacao = pd.DataFrame(
    validacao
)

df_duplicados = (
    pd.concat(
        duplicados,
        ignore_index=True,
    )
    if duplicados
    else pd.DataFrame()
)

df_validacao
